In [18]:
import pandas as pd

sales = pd.DataFrame({
    "salesperson": ["Rohit", "Rohit", "Rohit", "Amit", "Amit", "Neha", "Neha", "Neha"],
    "month": ["Jan", "Feb", "Mar", "Jan", "Feb", "Jan", "Feb", "Mar"],
    "region": ["North", "North", "North", "West", "West", "South", "South", "South"],
    "amount": [50000, 60000, 55000, 75000, 45000, 80000, 90000, 70000]
})
print(sales)

  salesperson month region  amount
0       Rohit   Jan  North   50000
1       Rohit   Feb  North   60000
2       Rohit   Mar  North   55000
3        Amit   Jan   West   75000
4        Amit   Feb   West   45000
5        Neha   Jan  South   80000
6        Neha   Feb  South   90000
7        Neha   Mar  South   70000


In [19]:
pivot = sales.pivot_table(
    index = "salesperson",
    columns = "month",
    values = "amount",
    aggfunc = "sum"
)
print(pivot)

month            Feb      Jan      Mar
salesperson                           
Amit         45000.0  75000.0      NaN
Neha         90000.0  80000.0  70000.0
Rohit        60000.0  50000.0  55000.0


In [20]:
sales.pivot_table(
    index = "salesperson",
    columns = "month",
    values = "amount",
    aggfunc = "sum",
    fill_value=0,
    margins= True
 )

month,Feb,Jan,Mar,All
salesperson,,,,
Amit,45000,75000,0,120000
Neha,90000,80000,70000,240000
Rohit,60000,50000,55000,165000
All,195000,205000,125000,525000


In [21]:
# Q1: Total amount by region (rows) and month (columns), with 0 for gaps
sales.pivot_table(
    index = "region",
    columns = "month",
    values = "amount",
    aggfunc = "sum",
    fill_value = 0,
    
 )

month,Feb,Jan,Mar
region,,,
North,60000,50000,55000
South,90000,80000,70000
West,45000,75000,0


In [22]:
# Q2: Same but with grand totals
sales.pivot_table(
    index = "region",
    columns = "month",
    values = "amount",
    aggfunc = "sum",
    fill_value = 0,
    margins = True
 )

month,Feb,Jan,Mar,All
region,,,,
North,60000,50000,55000,165000
South,90000,80000,70000,240000
West,45000,75000,0,120000
All,195000,205000,125000,525000


In [23]:
# Q3: COUNT of sales (not sum) by salesperson and month
sales.pivot_table(
    index = "salesperson",
    columns = "month",
    values = "amount",
    aggfunc = "count",
    fill_value=0,
    margins= True
 )

month,Feb,Jan,Mar,All
salesperson,,,,
Amit,1,1,0,2
Neha,1,1,1,3
Rohit,1,1,1,3
All,3,3,2,8


In [24]:
# Q4: Two aggregations at once
sales.pivot_table(
    index = "salesperson",
    columns = "month",
    values = "amount",
    aggfunc = ["sum", "count"],
    fill_value=0
 )

sum               count        
month          Feb    Jan    Mar   Feb Jan Mar
salesperson                                   
Amit         45000  75000      0     1   1   0
Neha         90000  80000  70000     1   1   1
Rohit        60000  50000  55000     1   1   1

In [27]:
# Add a duplicate: second Jan sale for Rohit
sales_dup = pd.concat([sales, pd.DataFrame({"salesperson": ["Rohit"], "month": ["Jan"], "region": ["North"], "amount": [20000]
                                           })], ignore_index = True)
#sales_dup.pivot(index="salesperson", columns= "month", values= "amount")

In [28]:
sales_dup.pivot_table(index="salesperson", columns="month", values="amount", aggfunc="sum")

month,Feb,Jan,Mar
salesperson,,,
Amit,45000.0,75000.0,NaN
Neha,90000.0,80000.0,70000.0
Rohit,60000.0,70000.0,55000.0


In [29]:
wide = pd.DataFrame({
    "salesperson": ["Rohit", "Amit", "Neha"],
    "Jan": [50000, 75000, 80000],
    "Feb": [60000, 45000, 90000],
    "Mar": [55000, 0, 70000]
})
print(wide)

  salesperson    Jan    Feb    Mar
0       Rohit  50000  60000  55000
1        Amit  75000  45000      0
2        Neha  80000  90000  70000


In [30]:
long = wide.melt(
    id_vars = "salesperson",
    var_name = "month",
    value_name = "amount"
)
print(long)

  salesperson month  amount
0       Rohit   Jan   50000
1        Amit   Jan   75000
2        Neha   Jan   80000
3       Rohit   Feb   60000
4        Amit   Feb   45000
5        Neha   Feb   90000
6       Rohit   Mar   55000
7        Amit   Mar       0
8        Neha   Mar   70000


In [31]:
long.groupby("month")["amount"].sum()

month
Feb    195000
Jan    205000
Mar    125000
Name: amount, dtype: int64

In [32]:
long[long["amount"] > 60000]

,salesperson,month,amount
1,Amit,Jan,75000
2,Neha,Jan,80000
5,Neha,Feb,90000
8,Neha,Mar,70000


In [33]:
long.pivot_table(index="salesperson", columns="month", values="amount")

month,Feb,Jan,Mar
salesperson,,,
Amit,45000.0,75000.0,0.0
Neha,90000.0,80000.0,70000.0
Rohit,60000.0,50000.0,55000.0


In [34]:
wide_with_region = pd.DataFrame({
    "salesperson": ["Rohit", "Amit"],
    "region": ["North", "West"],
    "Jan": [50000, 75000],
    "Feb": [60000, 45000]
})

wide_with_region.melt(
    id_vars=["salesperson", "region"],   # both stay
    var_name="month",
    value_name="amount"
)

,salesperson,region,month,amount
0,Rohit,North,Jan,50000
1,Amit,West,Jan,75000
2,Rohit,North,Feb,60000
3,Amit,West,Feb,45000


In [35]:
# Q1: Melt the `wide` DataFrame to long. How many rows result? Predict first.
#     (Hint: 3 people × 3 months = ?)
wide.melt(id_vars="salesperson", var_name="month", value_name="amount")

,salesperson,month,amount
0,Rohit,Jan,50000
1,Amit,Jan,75000
2,Neha,Jan,80000
3,Rohit,Feb,60000
4,Amit,Feb,45000
5,Neha,Feb,90000
6,Rohit,Mar,55000
7,Amit,Mar,0
8,Neha,Mar,70000


In [36]:
# Q2: After melting, total sales per month
wide.melt(id_vars="salesperson", var_name="month", value_name="amount").groupby("month")["amount"].sum()

month
Feb    195000
Jan    205000
Mar    125000
Name: amount, dtype: int64

In [37]:
# Q3: After melting, filter to sales above 60000
wide.melt(id_vars="salesperson", var_name="month", value_name="amount").query("amount > 60000")

,salesperson,month,amount
1,Amit,Jan,75000
2,Neha,Jan,80000
5,Neha,Feb,90000
8,Neha,Mar,70000


In [38]:
# Q4: The round trip — melt wide to long, then pivot_table back to wide. 
#     Do you get the original back?
long_again = wide.melt(id_vars="salesperson", var_name="month", value_name="amount")
back_to_wide = long_again.pivot_table(index="salesperson", columns="month", values="amount")
print(back_to_wide)

month            Feb      Jan      Mar
salesperson                           
Amit         45000.0  75000.0      0.0
Neha         90000.0  80000.0  70000.0
Rohit        60000.0  50000.0  55000.0
